# Reasoning Models and Test-Time Compute

The runnable code of [Reasoning Models and Test-Time Compute](https://residual-stream.lvtan71.workers.dev/posts/reasoning-and-test-time-compute/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/reasoning-and-test-time-compute.ipynb)

**Hardware:** Use a GPU runtime (Colab: Runtime → Change runtime type → T4 GPU). About 86 min end to end.
Downloads Qwen2.5-1.5B-Instruct (about 3 GB).

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "reasoning-and-test-time-compute"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in ['shared/reasoning-and-test-time-compute/chain.py', 'shared/reasoning-and-test-time-compute/lm.py', 'shared/reasoning-and-test-time-compute/tts.py']:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## What thinking costs

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/reasoning-and-test-time-compute/#what-thinking-costs)

In [ ]:
# What thinking tokens cost, for one request on two real configurations
# (from their config.json files):
#   Qwen2.5-32B: 64 layers, 8 key/value heads of 128 (GQA, "The KV Cache"), BF16;
#   DeepSeek-R1 (671B, 37B active): 61 layers of multi-head latent
#   attention, caching a 512-number latent plus a 64-number RoPE key per
#   token per layer ("Anatomy of a Frontier Model"), BF16.
# Decode time, batch 1, bandwidth-bound ("Small and Specialized Models"): each token reads the
# active weights plus the whole KV cache so far, at one H100's 3.35 TB/s.
# Qwen2.5-32B fits on one H100; DeepSeek-R1 needs many GPUs to hold its
# 671B parameters, so its row only illustrates the reading: its active
# weights and cache at one GPU's bandwidth, without the communication
# between GPUs that a real deployment adds ("Mixture of Experts").
H100 = 3.35e12
models = {
    "Qwen2.5-32B": dict(kv=2 * 64 * 8 * 128 * 2, weights=32.5e9 * 2),
    "DeepSeek-R1": dict(kv=61 * (512 + 64) * 2, weights=37e9 * 1),   # FP8 weights, the format it was trained in
}
print(f"{'':12s} {'KV per token':>13s}   {'thinking tokens':>15s} {'KV cache':>9s} {'decode time':>12s} {'avg tokens/s':>13s}")
for name, m in models.items():
    for n in (1_000, 10_000, 50_000):
        # time = sum over tokens of (weights + KV so far) / bandwidth
        seconds = (n * m["weights"] + m["kv"] * n * (n - 1) / 2) / H100
        print(f"{name:12s} {m['kv'] / 1024:9.1f} KiB   {n:15,d} {m['kv'] * n / 2**30:7.2f} GiB {seconds:10.1f} s {n / seconds:13.0f}")

The post printed:

```text
              KV per token   thinking tokens  KV cache  decode time  avg tokens/s
Qwen2.5-32B      256.0 KiB             1,000    0.24 GiB       19.4 s            51
Qwen2.5-32B      256.0 KiB            10,000    2.44 GiB      197.9 s            51
Qwen2.5-32B      256.0 KiB            50,000   12.21 GiB     1068.0 s            47
DeepSeek-R1       68.6 KiB             1,000    0.07 GiB       11.1 s            90
DeepSeek-R1       68.6 KiB            10,000    0.65 GiB      111.5 s            90
DeepSeek-R1       68.6 KiB            50,000    3.27 GiB      578.5 s            86
```

## An experiment: thinking on checkable problems

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/reasoning-and-test-time-compute/#an-experiment-thinking-on-checkable-problems)

In [ ]:
%%writefile tts.py
import re
import sys
import time
from collections import Counter
import numpy as np
import torch
from lm import model, tok, DEVICE

# Test-time compute on a problem set whose answers can be checked: 100
# three-step arithmetic word problems generated from templates, each with a
# known integer answer. Qwen2.5-1.5B-Instruct answers each one
#   direct:  "answer with just the number", greedy;
#   chain:   "think step by step, then 'Answer: <number>'", greedy;
#   sampled: the same chain prompt, 16 samples at temperature 0.7, scored
#            one at a time and by majority vote over the first k.
# Every decoding parameter is passed explicitly ("Decoding").
seed = int(sys.argv[1]) if len(sys.argv) > 1 else 0
rng = np.random.default_rng(1234)                                   # the problems are fixed across seeds
TEMPLATES = [
    ("A library has {a} shelves with {b} books on each. It lends out {c} books and receives {d} new ones. How many books does it have now?", lambda a, b, c, d: a * b - c + d),
    ("Minh buys {a} packs of {b} stickers, gives {c} stickers to each of his {d} friends, and keeps the rest. How many does he keep?", lambda a, b, c, d: a * b - c * d),
    ("A train has {a} cars with {b} seats each. {c} seats are broken, and the remaining seats are shared equally by {d} tour groups. How many seats does each group get?", None),
    ("A farm collects {a} eggs a day for {b} days, then sells them in cartons of {d}. {c} eggs break first. How many full cartons can it sell?", lambda a, b, c, d: (a * b - c) // d),
]
problems = []
while len(problems) < 100:
    k = len(problems) % len(TEMPLATES)
    text, f = TEMPLATES[k]
    a, b, c, d = (int(x) for x in rng.integers([3, 4, 2, 2], [15, 25, 30, 9]))
    if k == 2:                                                      # make the division exact
        if (a * b - c) % d or a * b <= c:
            continue
        f = lambda a, b, c, d: (a * b - c) // d
    answer = f(a, b, c, d)
    if answer <= 0:
        continue
    problems.append((text.format(a=a, b=b, c=c, d=d), answer))

m = model()
torch.manual_seed(seed)
def chat(content):
    return tok.apply_chat_template([{"role": "user", "content": content}], add_generation_prompt=True,
                                   return_tensors="pt", return_dict=True)["input_ids"].to(DEVICE)
def number(text):
    after = text.split("Answer:")[-1]
    nums = re.findall(r"-?\d[\d,]*", after)
    return int(nums[-1].replace(",", "")) if nums else None

GEN = dict(do_sample=False, repetition_penalty=1.0, top_k=None, top_p=None, temperature=None, max_new_tokens=320)
SAMPLE = dict(do_sample=True, temperature=0.7, top_p=1.0, top_k=0, repetition_penalty=1.0, max_new_tokens=320,
              num_return_sequences=16)
res = {"direct": [], "chain": [], "samples": []}
tokens = {"direct": [], "chain": [], "samples": []}
t0 = time.perf_counter()
for q, ans in problems:
    if seed == 0:                                                   # greedy answers don't depend on the seed
        for kind, prompt in (("direct", q + " Answer with just the number."),
                             ("chain", q + " Think step by step, then give the final answer as 'Answer: <number>'.")):
            x = chat(prompt)
            out = m.generate(x, **GEN)[0, x.shape[1]:]
            res[kind].append(number(tok.decode(out, skip_special_tokens=True)) == ans)
            tokens[kind].append(int((out != tok.pad_token_id).sum()))
    x = chat(q + " Think step by step, then give the final answer as 'Answer: <number>'.")
    outs = m.generate(x, **SAMPLE)[:, x.shape[1]:]
    res["samples"].append([number(tok.decode(o, skip_special_tokens=True)) for o in outs])
    tokens["samples"].append([int((o != tok.pad_token_id).sum()) for o in outs])
print(f"seed {seed}: {len(problems)} problems, {time.perf_counter() - t0:.0f} s")
if seed == 0:
    for kind in ("direct", "chain"):
        print(f"  greedy, {kind:6s}: {np.mean(res[kind]):.0%} right, {np.mean(tokens[kind]):.0f} tokens per answer")
S, truth = res["samples"], [a for _, a in problems]
print(f"  sampled chain, one sample: {np.mean([[s == t for s in ss] for ss, t in zip(S, truth)]):.0%} right, "
      f"{np.mean(tokens['samples']):.0f} tokens per answer")
for k in (1, 2, 4, 8, 16):
    vote = [Counter([s for s in ss[:k] if s is not None]).most_common(1) for ss in S]
    acc = np.mean([bool(v) and v[0][0] == t for v, t in zip(vote, truth)])
    any_right = np.mean([t in ss[:k] for ss, t in zip(S, truth)])
    print(f"  majority of {k:2d}: {acc:.0%} right ({k * np.mean(tokens['samples']):5.0f} tokens per question); "
          f"some sample right: {any_right:.0%}")

In [ ]:
# The post ran two seeds:
%run tts.py 0
%run tts.py 1

The post printed:

```text
seed 0: 100 problems, 2570 s
  greedy, direct: 1% right, 3 tokens per answer
  greedy, chain : 100% right, 215 tokens per answer
  sampled chain, one sample: 97% right, 178 tokens per answer
  majority of  1: 97% right (  178 tokens per question); some sample right: 97%
  majority of  2: 98% right (  357 tokens per question); some sample right: 100%
  majority of  4: 100% right (  713 tokens per question); some sample right: 100%
  majority of  8: 100% right ( 1426 tokens per question); some sample right: 100%
  majority of 16: 100% right ( 2853 tokens per question); some sample right: 100%
seed 1: 100 problems, 1828 s
  sampled chain, one sample: 98% right, 175 tokens per answer
  majority of  1: 99% right (  175 tokens per question); some sample right: 99%
  majority of  2: 99% right (  351 tokens per question); some sample right: 100%
  majority of  4: 100% right (  702 tokens per question); some sample right: 100%
  majority of  8: 100% right ( 1404 tokens per question); some sample right: 100%
  majority of 16: 100% right ( 2808 tokens per question); some sample right: 100%
```

In [ ]:
%%writefile tts_hard.py
import os
import sys
from collections import Counter
import numpy as np
import torch
from lm import model, tok, DEVICE
sys.path.insert(0, "../post33")

# The same measurement on harder problems, where one chain isn't enough:
# "Chain-of-Thought and Prompt Chaining"'s serial arithmetic with eight dependent operations (40
# problems), on which one greedy chain is right 92% of the time. 16
# chains sampled at temperature 0.7 per problem; the majority answer of
# the first k, and whether any of them is right (what a perfect verifier
# could pick, best-of-N).
src = open("chain.py").read()
exec(src[src.index("N = int("):src.index("m = model()")])          # the problem generator of "Chain-of-Thought and Prompt Chaining", same seed
probs = problems[8]
m = model()
SAMPLE = dict(do_sample=True, temperature=0.7, top_p=1.0, top_k=0, repetition_penalty=1.0)
def number(text):
    if "Answer:" not in text:
        return None
    import re
    nums = re.findall(r"-?\d+", text.split("Answer:")[-1].replace(",", ""))
    return int(nums[-1]) if nums else None
torch.manual_seed(0)
answers, tokens = [], []
for start, steps, answer in probs:
    prompt = (f"Start with {start}. " + " ".join(f"Then {s}." for s, _, _ in steps) +
              " Work it out step by step, then give the final result as 'Answer: <number>'.")
    x = tok.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True,
                                return_tensors="pt", return_dict=True)["input_ids"].to(DEVICE)
    out = m.generate(x.repeat(16, 1), max_new_tokens=380, **SAMPLE)
    gen = out[:, x.shape[1]:]
    answers.append([number(tok.decode(g, skip_special_tokens=True)) for g in gen])
    tokens.append(int((gen != tok.pad_token_id).sum()) / 16)
    print(f"problem {len(answers)}: {sum(a == answer for a in answers[-1])}/16 samples right", flush=True)
truth = [a for _, _, a in probs]
print(f"\n{len(probs)} problems, eight operations each; {np.mean(tokens):.0f} tokens per sample")
for k in (1, 2, 4, 8, 16):
    maj = [Counter(x for x in a[:k] if x is not None).most_common(1) for a in answers]
    right = np.mean([bool(mc) and mc[0][0] == t for mc, t in zip(maj, truth)])
    some = np.mean([t in a[:k] for a, t in zip(answers, truth)])
    print(f"  majority of {k:2d}: {right:4.0%} right ({k * np.mean(tokens):6.0f} tokens per question); "
          f"some sample right: {some:4.0%}")
per = [sum(x == t for x in a) / 16 for a, t in zip(answers, truth)]
print(f"problems by share of samples right: none {sum(p == 0 for p in per)}, under half {sum(0 < p < 0.5 for p in per)}, "
      f"half or more {sum(0.5 <= p < 1 for p in per)}, all {sum(p == 1 for p in per)}")

In [ ]:
# The post saved this run's output for the next cell to parse:
!{sys.executable} tts_hard.py | tee tts_hard.out

The post printed:

```text

40 problems, eight operations each; 202 tokens per sample
  majority of  1:  82% right (   202 tokens per question); some sample right:  82%
  majority of  2:  88% right (   403 tokens per question); some sample right:  90%
  majority of  4:  92% right (   807 tokens per question); some sample right:  95%
  majority of  8:  92% right (  1614 tokens per question); some sample right:  95%
  majority of 16:  92% right (  3227 tokens per question); some sample right:  98%
problems by share of samples right: none 1, under half 2, half or more 25, all 12
```

In [ ]:
import re
from math import comb

# From tts_hard.out's per-problem counts (c of n = 16 samples right):
# the average accuracy of one sample, and the unbiased estimate of the
# chance that at least one of k samples is right (pass@k, Chen et al.,
# 2021): 1 - C(n - c, k) / C(n, k), averaged over problems. That is what a
# perfect verifier choosing among k samples would score.
counts = [int(m) for m in re.findall(r"^problem \d+: (\d+)/16", open("tts_hard.out").read(), re.M)]
n = 16
print(f"{len(counts)} problems; one sample right on average: {sum(counts) / (n * len(counts)):.1%}")
for k in (1, 2, 4, 8, 16):
    p = sum(1 - comb(n - c, k) / comb(n, k) for c in counts) / len(counts)
    print(f"  pass@{k:<2d} (some of {k:2d} samples right): {p:.1%}")

The post printed:

```text
40 problems; one sample right on average: 86.6%
  pass@1  (some of  1 samples right): 86.6%
  pass@2  (some of  2 samples right): 93.1%
  pass@4  (some of  4 samples right): 94.9%
  pass@8  (some of  8 samples right): 96.2%
  pass@16 (some of 16 samples right): 97.5%
```